# Cheat Sheet 11: Predicting Data with Test/Train Splits

_MMA 860 – Management of Data_

## What this covers

- Why we split data into training and test sets
- Using `train_test_split` from scikit-learn
- Comparing R², RMSE, and MAE between train and test
- Time series caveat: never randomize a temporal split

## Working with AI

AI is useful here for one thing in particular: explaining *why* train and test scores differ. Paste both sets of metrics and ask *"What does this gap tell me about my model?"*

Be cautious: AI will sometimes recommend cross-validation, regularization, or ensemble methods that are out of scope for this course. Stay focused on the linear regression workflow until later courses introduce those tools.

## A quick mental model

We split the data so that we test the model on observations it has never seen. If the model performs much worse on test than train, it has overfit. If it performs about the same, it has generalized.

Three rules to remember:

1. Train and test sets must be **mutually exclusive**.
2. They should come from the **same distribution** – use random sampling.
3. For **time series**, always test on the most recent data. Random splits leak the future into the past.

# Tasks

## Create test and train data sets

In [20]:
#Import Grocery Data

import pandas as pd
import os
from os.path import curdir
path = os.path.join(curdir,'Data',"MMA_860_Grocery_Data.csv")
data = pd.read_csv(path,index_col="Obs")
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 1 to 1000
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Grocery_Bill       1000 non-null   str  
 1   N_Adults           1000 non-null   int64
 2   Family_Income      1000 non-null   str  
 3   Family_Size        1000 non-null   int64
 4   N_Vehicles         1000 non-null   int64
 5   Distance_to_Store  1000 non-null   int64
 6   Vegetarian         1000 non-null   int64
 7   N_Children         1000 non-null   int64
 8   Family_Pet         1000 non-null   int64
dtypes: int64(7), str(2)
memory usage: 70.4 KB


In [23]:
# Convert the data types of the columns to numeric, if they are not already
# Remove dollar signs and commas from Grocery_Bill and Family_Income columns
data['Grocery_Bill'] = data['Grocery_Bill'].replace('[$,]', '', regex=True)
data['Family_Income'] = data['Family_Income'].replace('[$,]', '', regex=True)
data = data.apply(pd.to_numeric, errors='coerce')

In [24]:
data.head()

,Grocery_Bill,N_Adults,Family_Income,Family_Size,N_Vehicles,Distance_to_Store,Vegetarian,N_Children,Family_Pet
Obs,,,,,,,,,
1,357.73,2,142141,4,3,15,0,2,1
2,276.84,2,145916,2,1,4,0,0,0
3,197.92,1,86185,1,2,14,0,0,0
4,315.75,2,145998,3,1,8,0,1,0
5,202.89,1,79341,1,2,19,1,0,0


In [25]:
#Set X to be all values except Grocery_Bill and vice-versa for y
X = data.drop(columns=['Grocery_Bill']).values
y = data['Grocery_Bill'].values

Depending on the size of your dataset you can evaluate how large a test set is practical. The larger the dataset, the larger your test data can be. In this case, we will use 30% for testing, and 70% for training. This is probably a good rule of thumb. Create these datasets under the names ‘test’ and ‘train’.

Note: there are three important things to keep in mind:
1. Test and train sets must be mutually exclusive (i.e., no overlapping data)
2. Test and train sets must contain the same pattern of data (i.e., you should same randomly)
3. If you have time series data, you should always test on the most recent data

To sample randomly without replacement, you could use the following code. It will take a 70% train sample and a 30% test sample:

In [26]:
'''
Scikit learn has a built-in function for splitting data into training 
and testing datasets. Here we specify the X array, y array and train_size.
Setting a random_state makes our results reproducible.
'''
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,y,train_size=0.7,random_state=1)

In [27]:
# Train our model and assess it against training data
from sklearn.linear_model import LinearRegression
reg = LinearRegression().fit(X_train, y_train)

## Predict Values

The ‘predict’ function in Scikit allows you to use the linear regression model to predict values (in this case, the grocery bill). You can choose the dataset on which you would like to predict. The resulting array will contain the predicted values. The code looks like this:

In [28]:
#I have suppressed the output to the first 5 numbers only
#For the whole array, remove the appended '[0:5]'
print(reg.predict(X_test)[0:5])

[170.39319004 243.81684043 172.68059222 161.39855339 164.97620406]


## Calculate Statistics

For test & train comparisons, you will now have to calculate some of the statistics we use to validate model accuracy: the $R^2$ for test data, RMSE (Root Mean Squared Error), and MAE (Mean Absolute Error). To calculate these we will need to import some additional functions from sklearn.

In [29]:
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from numpy import sqrt

### Train Data

In [30]:
print("R^2:",reg.score(X_train,y_train))
print("Root Mean Squared Error:",sqrt(
    mean_squared_error(y_train,reg.predict(X_train))))
print("Mean Absolute Error:",mean_absolute_error(
    y_train,reg.predict(X_train))) 

R^2: 0.8472759620602488
Root Mean Squared Error: 31.7387681032616
Mean Absolute Error: 23.627633776899472


### Test Data 

In [31]:
print("R^2:",reg.score(X_test,y_test))
print("Root Mean Squared Error:",sqrt(
    mean_squared_error(y_test,reg.predict(X_test))))
print("Mean Absolute Error:",mean_absolute_error(
    y_test,reg.predict(X_test))) 

R^2: 0.8185994096384803
Root Mean Squared Error: 34.37801803441695
Mean Absolute Error: 25.192646345546713


We can compare our $R^2$ values and RMSEs directly. RMSE tends to be a more reliable measure of fit, especially when you would like to penalize large errors. Usually, we expect our model to perform worse on the test set. In this case, model performance is very similar – this is a good thing! 

## Try it yourself

1. Try different train/test splits (50/50, 80/20, 90/10). How stable are the test metrics?
2. Try different `random_state` values. How much does R² move?
3. **AI exercise.** Ask your AI tool: *"My test R² is much lower than my train R². What could be causing this and how would I diagnose it?"* Compare its list to the regression assumptions we covered earlier.